In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- CONFIGURATION ---
EPSILON = 1e-6 
N_PCA = 3 # Aim for 3 principal components

# 1. LOAD DATA
file_path = 'USD_Economy_Centurion_Real_Data_v2_PROCESSED.csv'
df = pd.read_csv(file_path)
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df_features = pd.DataFrame(index=df.index)

# 2. FEATURE ENGINEERING
if 'US_NFP_Payrolls' in df.columns:
    df_features['NFP_Surprise'] = df['US_NFP_Payrolls'] - df['US_NFP_Payrolls'].rolling(12).mean()
    df_features['NFP_Momentum'] = df['US_NFP_Payrolls'].pct_change(1)
if 'US_Unemployment_Rate' in df.columns:
    df_features['Unemp_Delta'] = df['US_Unemployment_Rate'].diff(1)
    # Sahm_Signal assumed to be in the raw data or engineered elsewhere
    if 'US_Sahm_Rule' in df.columns:
        df_features['Sahm_Signal'] = df['US_Sahm_Rule']
if 'US_Hourly_Earnings' in df.columns:
    df_features['Wage_Growth_YoY'] = df['US_Hourly_Earnings'].pct_change(252) # Daily proxy for YoY
if 'US_JOLTS_Openings' in df.columns and 'US_Unemployment_Rate' in df.columns:
    ratio = df['US_JOLTS_Openings'] / (df['US_Unemployment_Rate'] + EPSILON)
    df_features['Vacancy_Ratio_Log'] = np.log(np.maximum(ratio, EPSILON))

# 3. CLEANING & SCALING
df_features.replace([np.inf, -np.inf], np.nan, inplace=True)
df_features.dropna(inplace=True)
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=df_features.columns, index=df_features.index)

# 4. PCA REDUCTION
pca = PCA(n_components=N_PCA)
df_pca = pd.DataFrame(pca.fit_transform(df_scaled), index=df_scaled.index, columns=[f'EMP_PCA_{i+1}' for i in range(N_PCA)])

# 5. SAVE
output_file = 'PCA_EMPLOYMENT_FEATURES.csv'
df_pca.to_csv(output_file)
print(f"SUCCESS! Saved {df_pca.shape[1]} PCA components for Employment to {output_file}")

SUCCESS! Saved 3 PCA components for Employment to PCA_EMPLOYMENT_FEATURES.csv


In [2]:
# Notebook 2: Inflation Regimes
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
EPSILON = 1e-6; N_PCA = 3
file_path = 'USD_Economy_Centurion_Real_Data_v2_PROCESSED.csv'
df = pd.read_csv(file_path); df['Date'] = pd.to_datetime(df['Date']); df.set_index('Date', inplace=True)
df_features = pd.DataFrame(index=df.index)

# FEATURE ENGINEERING
if 'US_CPI_Headline' in df.columns:
    cpi_yoy = df['US_CPI_Headline'].pct_change(252)
    df_features['CPI_Acceleration'] = cpi_yoy.diff(1).diff(1)
    df_features['Real_CPI_Trend'] = cpi_yoy
if 'US_PCE_Core' in df.columns:
    df_features['PCE_Delta'] = df['US_PCE_Core'].diff(1)
    df_features['PCE_Trend_Dev'] = df['US_PCE_Core'] - df['US_PCE_Core'].rolling(6).mean()
if 'US_Inflation_Exp_10Y' in df.columns:
    df_features['Exp_Breakout'] = df['US_Inflation_Exp_10Y'] - df['US_Inflation_Exp_10Y'].rolling(50).mean()

# CLEANING, SCALING, PCA, AND SAVING
df_features.replace([np.inf, -np.inf], np.nan, inplace=True); df_features.dropna(inplace=True)
scaler = StandardScaler(); df_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=df_features.columns, index=df_features.index)
pca = PCA(n_components=N_PCA); df_pca = pd.DataFrame(pca.fit_transform(df_scaled), index=df_scaled.index, columns=[f'INF_PCA_{i+1}' for i in range(N_PCA)])
df_pca.to_csv('PCA_INFLATION_FEATURES.csv')
print(f"SUCCESS! Saved {df_pca.shape[1]} PCA components for Inflation to PCA_INFLATION_FEATURES.csv")

SUCCESS! Saved 3 PCA components for Inflation to PCA_INFLATION_FEATURES.csv


In [3]:
# Notebook 3: Growth Regimes
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
EPSILON = 1e-6; N_PCA = 3
file_path = 'USD_Economy_Centurion_Real_Data_v2_PROCESSED.csv'
df = pd.read_csv(file_path); df['Date'] = pd.to_datetime(df['Date']); df.set_index('Date', inplace=True)
df_features = pd.DataFrame(index=df.index)

# FEATURE ENGINEERING
if 'US_GDP_Real' in df.columns:
    df_features['GDP_Velocity'] = df['US_GDP_Real'].diff(1)
if 'US_Leading_Index' in df.columns:
    df_features['LEI_Signal'] = df['US_Leading_Index'].diff(1)
if 'US_Philly_Fed_Mfg' in df.columns:
    roll_mean = df['US_Philly_Fed_Mfg'].rolling(60).mean()
    roll_std = df['US_Philly_Fed_Mfg'].rolling(60).std()
    df_features['Mfg_Sentiment_Z'] = (df['US_Philly_Fed_Mfg'] - roll_mean) / (roll_std + EPSILON)

# CLEANING, SCALING, PCA, AND SAVING
df_features.replace([np.inf, -np.inf], np.nan, inplace=True); df_features.dropna(inplace=True)
scaler = StandardScaler(); df_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=df_features.columns, index=df_features.index)
pca = PCA(n_components=N_PCA); df_pca = pd.DataFrame(pca.fit_transform(df_scaled), index=df_scaled.index, columns=[f'GRW_PCA_{i+1}' for i in range(N_PCA)])
df_pca.to_csv('PCA_GROWTH_FEATURES.csv')
print(f"SUCCESS! Saved {df_pca.shape[1]} PCA components for Growth to PCA_GROWTH_FEATURES.csv")

SUCCESS! Saved 3 PCA components for Growth to PCA_GROWTH_FEATURES.csv


In [4]:
# Notebook 4: Monetary Regimes
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
EPSILON = 1e-6; N_PCA = 3
file_path = 'USD_Economy_Centurion_Real_Data_v2_PROCESSED.csv'
df = pd.read_csv(file_path); df['Date'] = pd.to_datetime(df['Date']); df.set_index('Date', inplace=True)
df_features = pd.DataFrame(index=df.index)

# FEATURE ENGINEERING
if 'US_Fed_Funds_Daily' in df.columns:
    df_features['Policy_Step'] = df['US_Fed_Funds_Daily'].diff(20)
    df_features['Rate_ROC'] = df['US_Fed_Funds_Daily'].diff(60)
if 'US_Fed_Balance_Sheet' in df.columns:
    df_features['Liquidity_Impulse'] = df['US_Fed_Balance_Sheet'].pct_change(4)
    df_features['Liquidity_Trend'] = df['US_Fed_Balance_Sheet'].pct_change(252) # Daily proxy for YoY

# CLEANING, SCALING, PCA, AND SAVING
df_features.replace([np.inf, -np.inf], np.nan, inplace=True); df_features.dropna(inplace=True)
scaler = StandardScaler(); df_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=df_features.columns, index=df_features.index)
pca = PCA(n_components=N_PCA); df_pca = pd.DataFrame(pca.fit_transform(df_scaled), index=df_scaled.index, columns=[f'MON_PCA_{i+1}' for i in range(N_PCA)])
df_pca.to_csv('PCA_MONETARY_FEATURES.csv')
print(f"SUCCESS! Saved {df_pca.shape[1]} PCA components for Monetary to PCA_MONETARY_FEATURES.csv")

SUCCESS! Saved 3 PCA components for Monetary to PCA_MONETARY_FEATURES.csv


In [5]:
# Notebook 5: Consumer & Housing Regimes
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
EPSILON = 1e-6; N_PCA = 3
file_path = 'USD_Economy_Centurion_Real_Data_v2_PROCESSED.csv'
df = pd.read_csv(file_path); df['Date'] = pd.to_datetime(df['Date']); df.set_index('Date', inplace=True)
df_features = pd.DataFrame(index=df.index)

# FEATURE ENGINEERING
if 'US_Retail_Sales' in df.columns and 'US_CPI_Headline' in df.columns:
    nominal_chg = df['US_Retail_Sales'].pct_change(1)
    inflation = df['US_CPI_Headline'].pct_change(1)
    df_features['Retail_Real'] = nominal_chg - inflation
if 'US_UMich_Sentiment' in df.columns:
    df_features['Sentiment_Shock'] = df['US_UMich_Sentiment'].diff(1)
if 'US_Housing_Starts' in df.columns:
    df_features['Housing_MoM'] = df['US_Housing_Starts'].pct_change(1)

# CLEANING, SCALING, PCA, AND SAVING
df_features.replace([np.inf, -np.inf], np.nan, inplace=True); df_features.dropna(inplace=True)
scaler = StandardScaler(); df_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=df_features.columns, index=df_features.index)
pca = PCA(n_components=N_PCA); df_pca = pd.DataFrame(pca.fit_transform(df_scaled), index=df_scaled.index, columns=[f'CHS_PCA_{i+1}' for i in range(N_PCA)])
df_pca.to_csv('PCA_CONS_HSG_FEATURES.csv')
print(f"SUCCESS! Saved {df_pca.shape[1]} PCA components for Consumer/Housing to PCA_CONS_HSG_FEATURES.csv")

SUCCESS! Saved 3 PCA components for Consumer/Housing to PCA_CONS_HSG_FEATURES.csv


In [6]:
# Notebook 6: Financial Stress & FX/Global Regimes
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
EPSILON = 1e-6; N_PCA = 3
file_path = 'USD_Economy_Centurion_Real_Data_v2_PROCESSED.csv'
df = pd.read_csv(file_path); df['Date'] = pd.to_datetime(df['Date']); df.set_index('Date', inplace=True)
df_features = pd.DataFrame(index=df.index)

# FEATURE ENGINEERING
if 'US_Curve_10Y_2Y' in df.columns:
    df_features['Curve_Steepening'] = df['US_Curve_10Y_2Y'].diff(5)
if 'US_VIX' in df.columns:
    df_features['VIX_Spike'] = df['US_VIX'] / (df['US_VIX'].rolling(20).mean() + EPSILON)
if 'US_Financial_Stress' in df.columns:
    df_features['Stress_Velocity'] = df['US_Financial_Stress'].diff(5)
if 'EU_10Y_Bund_Yield' in df.columns:
    df_features['Bund_Momentum'] = df['EU_10Y_Bund_Yield'] - df['EU_10Y_Bund_Yield'].rolling(50).mean()
if 'EUR_USD_Exchange' in df.columns:
    # Robust Log Returns
    price_ratio = df['EUR_USD_Exchange'] / (df['EUR_USD_Exchange'].shift(1) + EPSILON)
    df_features['EURUSD_LogRet'] = np.log(np.maximum(price_ratio, EPSILON))
    ma50 = df['EUR_USD_Exchange'].rolling(50).mean()
    df_features['EURUSD_Dist_MA'] = (df['EUR_USD_Exchange'] - ma50) / (ma50 + EPSILON)

# CLEANING, SCALING, PCA, AND SAVING
df_features.replace([np.inf, -np.inf], np.nan, inplace=True); df_features.dropna(inplace=True)
scaler = StandardScaler(); df_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=df_features.columns, index=df_features.index)
pca = PCA(n_components=N_PCA); df_pca = pd.DataFrame(pca.fit_transform(df_scaled), index=df_scaled.index, columns=[f'SFX_PCA_{i+1}' for i in range(N_PCA)])
df_pca.to_csv('PCA_STRESS_FX_FEATURES.csv')
print(f"SUCCESS! Saved {df_pca.shape[1]} PCA components for Stress/FX to PCA_STRESS_FX_FEATURES.csv")

SUCCESS! Saved 3 PCA components for Stress/FX to PCA_STRESS_FX_FEATURES.csv


In [7]:
# Notebook 7: Divergence Regimes
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
EPSILON = 1e-6; N_PCA = 3
file_path = 'USD_Economy_Centurion_Real_Data_v2_PROCESSED.csv'
df = pd.read_csv(file_path); df['Date'] = pd.to_datetime(df['Date']); df.set_index('Date', inplace=True)
df_features = pd.DataFrame(index=df.index)

# FEATURE ENGINEERING
if 'US_Yield_10Y' in df.columns and 'EU_10Y_Bund_Yield' in df.columns:
    spread = df['US_Yield_10Y'] - df['EU_10Y_Bund_Yield']
    df_features['Spread_Delta'] = spread.diff(5)
    spread_mean = spread.rolling(60).mean(); spread_std = spread.rolling(60).std()
    df_features['Spread_ZScore'] = (spread - spread_mean) / (spread_std + EPSILON)

if 'US_Fed_Funds_Daily' in df.columns and 'EU_10Y_Bund_Yield' in df.columns:
    policy_gap = df['US_Fed_Funds_Daily'] - df['EU_10Y_Bund_Yield']
    df_features['Policy_Gap_Rate'] = policy_gap.pct_change(20)

if 'US_Yield_10Y' in df.columns and 'US_Inflation_Exp_10Y' in df.columns and 'EU_10Y_Bund_Yield' in df.columns:
    us_real = df['US_Yield_10Y'] - df['US_Inflation_Exp_10Y']
    eu_real_proxy = df['EU_10Y_Bund_Yield'] - 2.0 # Assuming 2% EU inflation target
    df_features['Real_Rate_Spread'] = us_real - eu_real_proxy

# CLEANING, SCALING, PCA, AND SAVING
df_features.replace([np.inf, -np.inf], np.nan, inplace=True); df_features.dropna(inplace=True)
scaler = StandardScaler(); df_scaled = pd.DataFrame(scaler.fit_transform(df_features), columns=df_features.columns, index=df_features.index)
pca = PCA(n_components=N_PCA); df_pca = pd.DataFrame(pca.fit_transform(df_scaled), index=df_scaled.index, columns=[f'DIV_PCA_{i+1}' for i in range(N_PCA)])
df_pca.to_csv('PCA_DIVERGENCE_FEATURES.csv')
print(f"SUCCESS! Saved {df_pca.shape[1]} PCA components for Divergence to PCA_DIVERGENCE_FEATURES.csv")

SUCCESS! Saved 3 PCA components for Divergence to PCA_DIVERGENCE_FEATURES.csv


In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. LIST ALL PCA FILES TO MERGE
pca_files = [
    'PCA_EMPLOYMENT_FEATURES.csv',
    'PCA_INFLATION_FEATURES.csv',
    'PCA_GROWTH_FEATURES.csv',
    'PCA_MONETARY_FEATURES.csv',
    'PCA_CONS_HSG_FEATURES.csv',
    'PCA_STRESS_FX_FEATURES.csv',
    'PCA_DIVERGENCE_FEATURES.csv'
]

# 2. LOAD AND MERGE ALL PCA COMPONENTS
print("Starting Master Merge of 7 PCA Files...")
merged_df = None
for f in pca_files:
    try:
        df_temp = pd.read_csv(f, index_col='Date')
        if merged_df is None:
            merged_df = df_temp
        else:
            merged_df = merged_df.join(df_temp, how='inner') # Inner join ensures only common dates remain
    except Exception as e:
        print(f"FATAL WARNING: Could not load or join {f}. Check the output of the corresponding notebook. Error: {e}")

if merged_df is None or merged_df.empty:
    print("FATAL ERROR: Merged DataFrame is empty. Cannot continue.")
else:
    # 3. FINAL CLEANING AND SCALING
    merged_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    merged_df.dropna(inplace=True)
    
    final_scaler = StandardScaler()
    df_final_scaled = pd.DataFrame(
        final_scaler.fit_transform(merged_df),
        columns=merged_df.columns,
        index=merged_df.index
    )

    # 4. SAVE FINAL FILE
    output_file = 'READY_FOR_DL_MASTER.csv'
    df_final_scaled.to_csv(output_file)

    print("\n---------------------------------------------------")
    print("SUCCESS! ALL REGIME FEATURES MERGED, PCA-REDUCED, AND SCALED.")
    print(f"Total Predictive Features: {df_final_scaled.shape[1]}") # 7 regimes * 3 components = 21 features
    print(f"Final File: {output_file}")
    print("This file is 100% ready for LSTM/Transformer training.")
    print("---------------------------------------------------")

Starting Master Merge of 7 PCA Files...

---------------------------------------------------
SUCCESS! ALL REGIME FEATURES MERGED, PCA-REDUCED, AND SCALED.
Total Predictive Features: 21
Final File: READY_FOR_DL_MASTER.csv
This file is 100% ready for LSTM/Transformer training.
---------------------------------------------------
